In [32]:
import dspy
import os
from dotenv import load_dotenv
from openinference.instrumentation.dspy import DSPyInstrumentor
from langfuse import get_client


load_dotenv("/home/azureuser/cloudfiles/code/email-generation-experiments/.env.local", override=True)

os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_BASE_URL"] = "http://localhost:3000"

langfuse = get_client()
DSPyInstrumentor().instrument()


Attempting to instrument while already instrumented


In [3]:
api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_base = os.getenv("AZURE_LANGUAGE_ENDPOINT")
api_version = os.getenv("AZURE_API_VERSION")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

lm_4o_mini = dspy.LM(
    model=f"azure/{os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')}",
    api_key=api_key,
    api_base=api_base,
    # api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0.8, 
    model_type='responses'
)

lm_5_mini = dspy.LM( 
    model=f"azure/gpt-5-mini",
    api_key=os.getenv("AZURE_OPENAI_4_1_KEY"),
    api_base=api_base,
    # api_version="2025-04-14",
    temperature=1, 
    model_type='responses'
)

lm_41_mini = dspy.LM(
    model=f"azure/gpt-4.1-mini",
    api_key=os.getenv("AZURE_OPENAI_4_1_KEY"),
    api_base=api_base,
    # api_version="2025-04-14",
    temperature=0.4, 
    model_type='responses'
)

# mistral = dspy.LM(
#     model="openai/mistral-small-2503",
#     api_key=os.getenv("AZURE_MISTRAL_KEY"),
#     api_base="https://anuj-ai-resource.services.ai.azure.com/openai/v1/",
#     api_version="2024-05-01-preview",
#     temperature=0.8, 
#     # model_type='responses'
# )

# phi4 = dspy.LM(
#     model="openai/Phi-4-reasoning",
#     api_key=os.getenv("AZURE_MISTRAL_KEY"),
#     api_base="https://anuj-ai-resource.services.ai.azure.com/openai/v1/",
#     api_version="2024-05-01-preview",
#     temperature=0.8, 
#     # model_type='responses'
# )

dspy.configure(lm=lm_41_mini)

In [4]:
import pandas as pd
from pathlib import Path

PREFIXES = [
    '/home/azureuser/cloudfiles/code/production-data/12-dec-25/tenant1-12-20-25',
]
OUTPUT_FILES = [f"{prefix}-MASTER-REPORT.csv" for prefix in PREFIXES]


def load_and_merge_csv(prefixes):
    """
    Load and normalize invoices, notes, activities, matters, clients, and messages
    from one or more dataset prefixes, then merge them by client_id.
    """
    if isinstance(prefixes, (str, Path)):
        prefixes = [str(prefixes)]
    prefixes = [str(p) for p in prefixes]

    file_suffix = {
        'invoices': '-invoices.csv',
        'activities': '-activities-REDACTED.csv',
        'notes': '-notes-REDACTED.csv',
        'matters': '-matters.csv',
        'clients': '-clients.csv',
        'messages': '-messages-REDACTED.csv',
    }
    buckets = {name: [] for name in file_suffix}

    for prefix in prefixes:
        for name, suffix in file_suffix.items():
            csv_path = Path(f'{prefix}{suffix}')
            if not csv_path.exists():
                if name == 'messages':
                    continue
                raise FileNotFoundError(f'Missing required file: {csv_path}')
            buckets[name].append(pd.read_csv(csv_path))

    def _concat(name):
        if not buckets[name]:
            return pd.DataFrame()
        return pd.concat(buckets[name], ignore_index=True)

    def _clean_string(series):
        out = series.astype('string').str.strip()
        return out.replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA, '<NA>': pd.NA})

    def _normalize_invoice_value(value):
        if pd.isna(value):
            return pd.NA
        s = str(value).strip()
        if not s:
            return pd.NA
        s = s.split('-')[0].replace(',', '').strip()
        try:
            return str(int(float(s)))
        except Exception:
            return s

    def _normalize_invoice_series(series):
        return series.map(_normalize_invoice_value).astype('string')

    def _normalize_common_ids(df):
        if 'client_id' in df.columns:
            df['client_id'] = pd.to_numeric(df['client_id'], errors='coerce').astype('Int64')
        if 'matter_id' in df.columns:
            df['matter_id'] = _clean_string(df['matter_id'])
        if 'invoice_number' in df.columns:
            df['invoice_number'] = _normalize_invoice_series(df['invoice_number'])

    def _parse_datetime_columns(df, columns):
        for col in columns:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce', utc=True, format='mixed')

    def _parse_numeric_columns(df, columns):
        for col in columns:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

    df_invoices = _concat('invoices')
    df_activities = _concat('activities')
    df_notes = _concat('notes')
    df_matters = _concat('matters')
    df_clients = _concat('clients')
    df_messages = _concat('messages')

    for frame in [df_invoices, df_activities, df_notes, df_matters, df_clients, df_messages]:
        _normalize_common_ids(frame)

    _parse_numeric_columns(df_invoices, ['invoice_amount', 'ar_amount', 'days_overdue'])
    if 'days_overdue' in df_invoices.columns:
        df_invoices['days_overdue'] = df_invoices['days_overdue'].astype('Int64')
    _parse_datetime_columns(df_invoices, ['submitted_on', 'due_on'])

    _parse_datetime_columns(df_activities, ['created_on'])
    _parse_datetime_columns(df_notes, ['created_on'])

    _parse_numeric_columns(
        df_matters,
        [
            'wip_amount',
            'payment_terms_in_days',
            'total_billed_amount',
            'total_collected_amount',
            'total_work_amount',
            'ar_total',
            'ar_overdue',
        ],
    )
    if 'payment_terms_in_days' in df_matters.columns:
        df_matters['payment_terms_in_days'] = pd.to_numeric(
            df_matters['payment_terms_in_days'], errors='coerce'
        ).astype('Int64')
    _parse_datetime_columns(df_matters, ['last_invoiced_on', 'last_paid_on', 'last_statement_sent_on'])

    _parse_numeric_columns(
        df_clients,
        [
            'wip_amount',
            'payment_terms_in_days',
            'total_billed_amount',
            'total_collected_amount',
            'total_work_amount',
            'ar_total',
            'ar_overdue',
        ],
    )
    if 'payment_terms_in_days' in df_clients.columns:
        df_clients['payment_terms_in_days'] = pd.to_numeric(
            df_clients['payment_terms_in_days'], errors='coerce'
        ).astype('Int64')
    _parse_datetime_columns(
        df_clients,
        ['last_invoiced_on', 'last_action_on', 'last_paid_on', 'last_statement_sent_on'],
    )

    _parse_datetime_columns(df_messages, ['sent_on'])

    if not df_invoices.empty:
        sort_col = 'submitted_on' if 'submitted_on' in df_invoices.columns else 'client_id'
        df_invoices = df_invoices.sort_values(sort_col, kind='stable')
        key_cols = [c for c in ['client_id', 'matter_id', 'invoice_number'] if c in df_invoices.columns]
        if key_cols:
            df_invoices = df_invoices.drop_duplicates(subset=key_cols, keep='last')

    if not df_matters.empty:
        sort_col = 'last_invoiced_on' if 'last_invoiced_on' in df_matters.columns else 'client_id'
        df_matters = df_matters.sort_values(sort_col, kind='stable')
        key_cols = [c for c in ['client_id', 'matter_id'] if c in df_matters.columns]
        if key_cols:
            df_matters = df_matters.drop_duplicates(subset=key_cols, keep='last')

    if not df_clients.empty:
        sort_col = 'last_action_on' if 'last_action_on' in df_clients.columns else 'client_id'
        df_clients = df_clients.sort_values(sort_col, kind='stable')
        if 'client_id' in df_clients.columns:
            df_clients = df_clients.drop_duplicates(subset=['client_id'], keep='last')

    if not df_notes.empty:
        dedupe_cols = [
            c
            for c in ['client_id', 'matter_id', 'invoice_number', 'created_on', 'content']
            if c in df_notes.columns
        ]
        if dedupe_cols:
            df_notes = df_notes.drop_duplicates(subset=dedupe_cols, keep='last')

    if not df_activities.empty:
        dedupe_cols = [
            c
            for c in [
                'client_id',
                'matter_id',
                'invoice_number',
                'created_on',
                'event_type',
                'name',
                'description',
            ]
            if c in df_activities.columns
        ]
        if dedupe_cols:
            df_activities = df_activities.drop_duplicates(subset=dedupe_cols, keep='last')

    if not df_messages.empty:
        dedupe_cols = [c for c in ['client_id', 'sent_on', 'subject', 'body'] if c in df_messages.columns]
        if dedupe_cols:
            df_messages = df_messages.drop_duplicates(subset=dedupe_cols, keep='last')

    def _records_by_client(df, value_name, sort_col=None):
        if df.empty or 'client_id' not in df.columns:
            return pd.DataFrame(columns=['client_id', value_name])

        value_cols = [c for c in df.columns if c != 'client_id']
        temp = df[['client_id'] + value_cols].copy()
        if sort_col and sort_col in temp.columns:
            temp = temp.sort_values(sort_col, kind='stable')

        if value_cols:
            grouped = (
                temp.groupby('client_id', dropna=False, sort=False)[value_cols]
                .apply(lambda g: g.to_dict('records'))
                .reset_index(name=value_name)
            )
        else:
            grouped = temp[['client_id']].drop_duplicates().reset_index(drop=True)
            grouped[value_name] = [[] for _ in range(len(grouped))]

        return grouped

    client_series = []
    for frame in [df_invoices, df_notes, df_activities, df_matters, df_clients, df_messages]:
        if 'client_id' in frame.columns and not frame.empty:
            client_series.append(frame['client_id'])

    if client_series:
        merged_by_client = pd.DataFrame(
            {
                'client_id': pd.concat(client_series, ignore_index=True)
                .dropna()
                .astype('Int64')
                .drop_duplicates()
            }
        )
    else:
        merged_by_client = pd.DataFrame(columns=['client_id'])

    if not df_clients.empty and 'client_id' in df_clients.columns:
        merged_by_client = merged_by_client.merge(df_clients, on='client_id', how='left')

    merged_by_client = merged_by_client.merge(
        _records_by_client(df_matters, 'matters', 'last_invoiced_on'), on='client_id', how='left'
    )
    merged_by_client = merged_by_client.merge(
        _records_by_client(df_invoices, 'invoices', 'submitted_on'), on='client_id', how='left'
    )
    merged_by_client = merged_by_client.merge(
        _records_by_client(df_notes, 'notes', 'created_on'), on='client_id', how='left'
    )
    merged_by_client = merged_by_client.merge(
        _records_by_client(df_activities, 'activities', 'created_on'), on='client_id', how='left'
    )
    merged_by_client = merged_by_client.merge(
        _records_by_client(df_messages, 'messages', 'sent_on'), on='client_id', how='left'
    )

    return (
        df_invoices.reset_index(drop=True),
        df_notes.reset_index(drop=True),
        df_activities.reset_index(drop=True),
        df_matters.reset_index(drop=True),
        df_clients.reset_index(drop=True),
        df_messages.reset_index(drop=True),
        merged_by_client.reset_index(drop=True),
    )


(
    df_invoices,
    df_notes,
    df_activities,
    df_matters,
    df_clients,
    df_messages,
    df_merged_client,
) = load_and_merge_csv(PREFIXES)

# Backward-compatible aliases used by downstream notebook cells.
df_invoice = df_invoices.copy()
df_note = df_notes.copy()
df_activity = df_activities.copy()
df_matter = df_matters.copy()
df_client = df_clients.copy()
df_message = df_messages.copy()



In [5]:
df_messages

,client_id,sent_on,subject,body
0,16,2025-09-26 14:35:04+00:00,RE: 2nd REQUEST: Legal invoice from <Organizat...,"Thank you again for all of your assistance, Jo..."
1,562116,2025-10-29 15:31:20+00:00,RE: Your October 2025 Invoice from <Organization>,"Invoice Submission (External)Hi <Person>, I ho..."
2,567518,2025-08-28 19:35:07+00:00,Re: **REPLY REQUESTED** <Organization> (056751...,[EXTERNAL EMAIL]\ndone!\napologies for the del...
3,567518,2025-08-28 12:34:41.720362+00:00,RE: **REPLY REQUESTED** <Organization>/<Organi...,Good morning <Person> Could I please ask you t...
4,580679,2025-12-02 19:08:27+00:00,Re: Husch Blackwell Invoice - <Organization>,[EXTERNAL EMAIL]\nSuper - thanks! <Person>. +4...


In [6]:
df_notes

,client_id,matter_id,invoice_number,created_on,content
0,16,<NA>,<NA>,2025-12-18 17:28:52.405655+00:00,<DateTime> <Person> sends MATTER level follow ...
1,16,<NA>,<NA>,2025-12-08 22:14:47.391040+00:00,<DateTime> <Person> sends MATTER level follow ...
2,16,<NA>,<NA>,2025-11-17 16:08:34.842470+00:00,<DateTime> <Person> (under 60 days) sends MATT...
3,16,<NA>,<NA>,2025-10-27 17:08:40.775347+00:00,10/27/25 <Person> (under 60 days) sends MATTER...
4,16,<NA>,<NA>,2025-10-08 03:58:14.830437+00:00,<DateTime> <Person> sends MATTER level follow ...
...,...,...,...,...,...
7151,6058243,<NA>,<NA>,2013-09-10 00:00:00+00:00,«Invoice_Days_Outstanding» 60 day call. 223 da...
7152,6058243,<NA>,<NA>,2013-09-09 00:00:00+00:00,No Text\nImported from 3E Collections
7153,6058243,6058243-0000001,<NA>,2025-10-06 16:09:53.526770+00:00,"<DateTime>: <Person>; UNA: $9,767.99; when pay..."
7154,6058243,6058243-0000020,<NA>,2025-10-14 21:39:47.448748+00:00,"10.14.25: <Person>; UNA: $9,767.99; when payme..."


In [7]:
df_summaries = pd.read_csv("/home/azureuser/cloudfiles/code/production-data/client_summarization_stored_completions_5th_feb26.csv")
df_summaries.head()

,completion_id,input_data,output_data
0,chatcmpl-D5OhEPBr9feteTKHFVqVPDGomJmFo,SyncCursorPage[ChatCompletionStoreMessage](dat...,"{\n ""id"": ""chatcmpl-D5OhEPBr9feteTKHFVqVPDGom..."
1,chatcmpl-D5Ew1ShgFECUweV0bORKx04cWf8uA,SyncCursorPage[ChatCompletionStoreMessage](dat...,"{\n ""id"": ""chatcmpl-D5Ew1ShgFECUweV0bORKx04cW..."
2,chatcmpl-D5EObN6Z1kzjxrZU2Mx7Wd0OkYq0p,SyncCursorPage[ChatCompletionStoreMessage](dat...,"{\n ""id"": ""chatcmpl-D5EObN6Z1kzjxrZU2Mx7Wd0Ok..."
3,chatcmpl-D5ENMYYiOXfT2iEM23SLnJ21at8M1,SyncCursorPage[ChatCompletionStoreMessage](dat...,"{\n ""id"": ""chatcmpl-D5ENMYYiOXfT2iEM23SLnJ21a..."
4,chatcmpl-D5BZKOuyVuimv1c0g4dKPJzdmTNZR,SyncCursorPage[ChatCompletionStoreMessage](dat...,"{\n ""id"": ""chatcmpl-D5BZKOuyVuimv1c0g4dKPJzdm..."


In [8]:
df_summaries.shape

(20, 3)

In [9]:
import ast
import json
import re

MSG_RE = re.compile(r"ChatCompletionStoreMessage\(content='((?:\\.|[^'])*)'.*?role='(.*?)'", re.DOTALL)

def _unescape_python_string(raw: str) -> str:
    try:
        return ast.literal_eval("'" + raw + "'")
    except Exception:
        return bytes(raw, 'utf-8').decode('unicode_escape', errors='replace')

def _split_user_content(content: str):
    if not content:
        return '', '', ''
    normalized = content.replace('\r\n', '\n')
    fin_key = 'Financial data:\n'
    notes_key = '\n\nNotes:\n'
    if fin_key not in normalized:
        return normalized.strip(), '', ''
    prompt, rest = normalized.split(fin_key, 1)
    financial_data = rest
    notes = ''
    if notes_key in rest:
        financial_data, notes = rest.split(notes_key, 1)
    return prompt.strip(), financial_data.strip(), notes.strip()

def parse_input_data(input_data: str):
    if not isinstance(input_data, str):
        return {'system_prompt': '', 'user_prompt': '', 'financial_data': '', 'notes': ''}
    messages = []
    for match in MSG_RE.finditer(input_data):
        raw = match.group(1)
        role = match.group(2)
        content = _unescape_python_string(raw)
        messages.append((role, content))
    system_prompt = '\n\n'.join([c for r, c in messages if r == 'system']).strip()
    user_content = '\n\n'.join([c for r, c in messages if r == 'user']).strip()
    user_prompt, financial_data, notes = _split_user_content(user_content)
    return {
        'system_prompt': system_prompt,
        'user_prompt': user_prompt,
        'financial_data': financial_data,
        'notes': notes,
    }

def parse_output_data(output_data: str) -> str:
    if not isinstance(output_data, str) or not output_data:
        return ''
    cleaned = re.sub(r':\s*<([^>]+)>', r': "<\1>"', output_data)
    try:
        payload = json.loads(cleaned)
        return payload.get('choices', [{}])[0].get('message', {}).get('content', '')
    except Exception:
        match = re.search(r'\"content\"\s*:\s*\"((?:\\.|[^\"])*)\"', output_data, re.DOTALL)
        if not match:
            return ''
        return bytes(match.group(1), 'utf-8').decode('unicode_escape', errors='replace')

parsed_rows = df_summaries['input_data'].apply(parse_input_data)
df_parsed = pd.DataFrame(list(parsed_rows))
df_extracted = pd.concat([
    df_summaries[['completion_id']].copy(),
    df_parsed,
    df_summaries['output_data'].apply(parse_output_data).rename('output_summary'),
], axis=1)


df_extracted.head()


,completion_id,system_prompt,user_prompt,financial_data,notes,output_summary
0,chatcmpl-D5OhEPBr9feteTKHFVqVPDGomJmFo,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 1 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,CreatedOn","The client currently has **$20,806.67** in ove..."
1,chatcmpl-D5Ew1ShgFECUweV0bORKx04cWf8uA,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 3 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,Create...","The client currently has **$16,373.00** in ove..."
2,chatcmpl-D5EObN6Z1kzjxrZU2Mx7Wd0OkYq0p,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 2 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,Create...","The client currently has **$6,096.75** in over..."
3,chatcmpl-D5ENMYYiOXfT2iEM23SLnJ21at8M1,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 2 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,Create...","The client currently has **$6,096.75** in over..."
4,chatcmpl-D5BZKOuyVuimv1c0g4dKPJzdmTNZR,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 3 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,Create...","The client currently has **$16,373.00** in ove..."


In [10]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")
df_extracted['notes_token_count'] = df_extracted['notes'].apply(lambda x: len(encoding.encode(x)))

In [11]:
# df_extracted.to_csv("/home/azureuser/cloudfiles/code/production-data/stored_completions_5-feb.csv", index = False)

In [12]:
notes = df_extracted.notes.tolist()

In [13]:
df_extracted.financial_data.tolist()[2]

'The client has 2 active invoices, lifetime billed amount of $10,037.91, lifetime collected amount of $3,941.16, WIP of $0.00, overdue AR of $6,096.75, outstanding AR of $6,096.75, average DSO of 260.'

In [14]:
df_extracted.notes.tolist()[1]

'ClientId,MatterId,InvoiceNumber,Content,CreatedOn\n033546,,,"Forwarded 1<DateTime> email to client re invoices <PhoneNumber>, <PhoneNumber> &amp; <PhoneNumber> and if we need to make further changes, in light of them paying the December and January invoices.",02/03/2026\n033546,,,"Forwarded 1/6 email to K. Borowik asking him to fix the address and I will send to client. Sent revised invoices to client, asking if that works for them.",<DateTime>\n033546,,,"Emailed K. Borowik asking him to revise invoices <PhoneNumber>, <PhoneNumber> &amp; 585916 and I will send to client asking if that format is acceptable. (Putting the property address in the client address.)",01/06/2026\n033546,,,"Emailed L. Schleicher re September invoices <PhoneNumber>, <PhoneNumber> L<Person> resubmitted <DateTime>, noting they paid a November invoice <DateTime>. Emailed client re same.",<DateTime>\n033546,,,Emailed L. Schleicher re client paid the October invoices leaving two September invoices open. <Person> rep

In [15]:
df_extracted.output_summary.tolist()[0]

'The client currently has **$20,806.67** in overdue accounts receivable with no payments collected or billed to date, indicating a significant delay in payment processing. There are no recent interactions or updates noted, suggesting a lack of follow-up on this outstanding invoice. Given the average Days Sales Outstanding (DSO) of **4**, immediate outreach is recommended to address the overdue balance and initiate payment discussions. It is crucial to monitor this account closely to prevent further delays and ensure timely resolution.'

In [16]:
import numpy as np

overlapping_invoices = np.intersect1d(df_notes.invoice_number, df_invoices.invoice_number.tolist()).tolist()
print("Total: ", len(overlapping_invoices), overlapping_invoices)

TypeError: boolean value of NA is ambiguous

In [34]:
import dspy
from typing import Literal, List

class ClientSummarization(dspy.Signature):
    """
    You are Oddr Summarizer, an AI assistant that helps law firms manage their Accounts Receivable. Your job is to generate clear, concise, and helpful summaries for attorneys using only the facts provided.

    You may infer logical next steps or patterns (e.g., payment gaps, lack of follow-up), but do not fabricate or invent any client interactions or internal updates that aren't explicitly present in the input.

    If no notes are available, avoid referring to interactions. Focus only on AR status and suggest possible outreach or monitoring based on the financial data.

    Your tone should be professional, factual, and efficient — like a trusted human AR professional writing a briefing. Favor direct, punchy sentences over formal or padded language.

    Given a client with the following financial data, notes and email conversations across the client, its matters, and invoices, please summarize the client in a way that will make it digestible and meaningful to the time-constrained Attorney responsible for the client so they can understand what is going on with their AR and who is responsible for any next steps.

    Frame this as a timeline, starting with an overview of the client's current AR status, then highlighting the most recent interaction and any key themes from earlier interactions — but only if such notes are present. Do not fabricate or infer specific updates. However, it's acceptable to suggest likely follow-up steps or highlight the absence of recent activity if that pattern is evident.

    Reduce repetitive details by grouping similar updates, eliminate specific payment amounts where appropriate, and keep the summary focused and efficient. Mention past events only if they are essential to understanding the current delay or next step. Avoid repeating older context that does not affect present action.

    Format the summary as a single paragraph without any headings or titles, and use smart **bolding** to highlight key points. 
    Limit the summary to **4 complete, efficient sentences**. Favor punchy, high-signal writing over formal explanations. 
    **Do not use headings, bullet points, or labels.** **Mention date always like Jan 01,2025**
    
    Structure the summary as a single flowing paragraph that follows this order:
    [AR overview]
    [Highlights from recent interactions — only if present]
    [Relevant context from earlier interactions — only if helpful]
    [Next steps or follow-up actions]
    """

    financial_data: str = dspy.InputField(
        desc="Structured csv containing Invoice Numbers, Dates, Amounts Outstanding, "
             "and total balance. These figures are immutable facts."
    )

    notes: str = dspy.InputField(
        desc="Mixed timeline of client interaction, internal attorney notes, and billing "
             "status updates. Use for context reasoning, personalization"
    ) 

    email_conversation: str = dspy.InputField(
        desc="Email exchanges between the client and the attorney/biller/collector. Use for context reasoning and personalization."
    )

    summary: str = dspy.OutputField(
        desc="Actionable summary of the Client across the invoices, notes and financial_data. Accurately mention dates, invoice numbers, amounts when summarizing notes and financial_data.  It should be helpful for the attorneys and provides a up-to-date detailed overview of the the account. "
    )


summarizer = dspy.Predict(ClientSummarization)
summarizer_cot = dspy.ChainOfThought(ClientSummarization)


In [35]:
import re


def data_formatter(df_invoices, df_notes, df_activities, invoice_number):

    notes = df_notes.query(f"invoice_number == {invoice_number}")[["created_on", "content"]].to_records(index=False)
    invoice_data = df_invoices.query(f"invoice_number=={invoice_number}").to_json(index=False, lines=True, orient='records')
    activities = (
        df_activities.query(f"invoice_number=={invoice_number}")
        .dropna()
        .drop(columns=['client_id', 'matter_id', 'invoice_number'])
        .to_markdown(index=False)
    )

    formatted_text = f"""
    ### INVOICE DATA:
    {invoice_data}

    ### NOTES:
    {notes}

    ### ACTIVITIES:
    {activities}
    """
    return formatted_text, notes, invoice_data, activities


def data_formatter_csv(df_invoices, df_notes, df_activities, invoice_number):

    notes = df_notes.query(f"invoice_number == {invoice_number}")[["created_on", "content"]].to_csv(index=False)
    invoice_data = df_invoices.query(f"invoice_number=='{invoice_number}'").to_csv(index=False)
    activities = (
        df_activities.query(f"invoice_number=='{invoice_number}'")
        .dropna()
        .drop(columns=['client_id', 'matter_id', 'invoice_number'])
        .to_csv(index=False)
    )

    formatted_text = f"""
    ## INVOICE_NUMBER: {invoice_number}
    ### INVOICE DATA:
    {invoice_data}

    ### NOTES:
    {notes}

    ### ACTIVITIES:
    {activities}

    //
    """
    return formatted_text, notes, invoice_data, activities


def data_formatter_csv_list(df_invoices, df_notes, df_activities, invoice_numbers):

    if not isinstance(invoice_numbers, list):
        invoice_numbers = [invoice_numbers]

    invoice_data = (
        df_invoices.query("invoice_number in @invoice_numbers")
        .drop(columns=['client_id', 'matter_id'])
        .to_csv(index=False)
    )

    activities = (
        df_activities.query("invoice_number in @invoice_numbers")
        .dropna()
        .drop(columns=['client_id', 'matter_id'])
        .to_csv(index=False)
    )

    notes = df_notes.query("invoice_number in @invoice_numbers")[["invoice_number", "created_on", "content"]].to_csv(index=False)
    formatted_text = f"""
    ### INVOICE DATA:
    {invoice_data}

    ### NOTES:
    {notes}

    ### ACTIVITIES:
    {activities}
    """
    return formatted_text, notes, invoice_data, activities


def _normalize_invoice_key(value):
    """Normalize invoice identifiers across files for safe joins."""
    if pd.isna(value):
        return pd.NA

    s = str(value).strip()
    if not s:
        return pd.NA

    s = s.split('-')[0].strip()
    numeric_candidate = s.replace(',', '')
    if re.fullmatch(r'\d+(?:\.0+)?', numeric_candidate):
        return str(int(float(numeric_candidate)))

    try:
        return str(int(float(numeric_candidate)))
    except Exception:
        return s


def _with_invoice_key(df, invoice_col='invoice_number'):
    out = df.copy()
    if invoice_col in out.columns:
        out['_invoice_key'] = out[invoice_col].map(_normalize_invoice_key)
    else:
        out['_invoice_key'] = pd.NA
    return out


def data_formatter_csv_list_v2(df_invoices, df_notes, df_activities, df_messages, invoice_numbers=None):
    """
    Robust invoice-level formatter that matches invoices, notes, activities and
    messages using normalized invoice keys and client_id relationships.

    Returns:
        formatted_text, notes_csv, invoice_data, activities_csv, messages_csv
    """
    if invoice_numbers is None:
        invoice_numbers = df_messages
        df_messages = pd.DataFrame(columns=['client_id', 'sent_on', 'subject', 'body'])

    if not isinstance(invoice_numbers, list):
        invoice_numbers = [invoice_numbers]

    target_keys = {_normalize_invoice_key(inv) for inv in invoice_numbers}
    target_keys = {k for k in target_keys if pd.notna(k)}

    invoices = _with_invoice_key(df_invoices)
    notes = _with_invoice_key(df_notes)
    activities = _with_invoice_key(df_activities)

    matched_invoices = invoices[invoices['_invoice_key'].isin(target_keys)].copy()

    target_client_ids = set()
    if 'client_id' in matched_invoices.columns:
        client_series = pd.to_numeric(matched_invoices['client_id'], errors='coerce').dropna().astype('Int64')
        target_client_ids = set(client_series.tolist())

    invoice_data_df = matched_invoices.drop(columns=['client_id', 'matter_id', '_invoice_key'], errors='ignore').copy()

    notes_mask = notes['_invoice_key'].isin(target_keys)
    if target_client_ids and 'client_id' in notes.columns:
        notes_client = pd.to_numeric(notes['client_id'], errors='coerce').astype('Int64')
        notes_mask = notes_mask & notes_client.isin(target_client_ids)

    notes_df = notes[notes_mask].drop(columns=['_invoice_key'], errors='ignore').copy()

    activities_mask = activities['_invoice_key'].isin(target_keys)
    if target_client_ids and 'client_id' in activities.columns:
        activities_client = pd.to_numeric(activities['client_id'], errors='coerce').astype('Int64')
        activities_mask = activities_mask & activities_client.isin(target_client_ids)

    activities_df = activities[activities_mask].drop(columns=['client_id', 'matter_id', '_invoice_key'], errors='ignore').copy()

    messages_df = df_messages.copy() if isinstance(df_messages, pd.DataFrame) else pd.DataFrame()
    if not messages_df.empty and 'client_id' in messages_df.columns and target_client_ids:
        messages_df['client_id'] = pd.to_numeric(messages_df['client_id'], errors='coerce').astype('Int64')
        messages_df = messages_df[messages_df['client_id'].isin(target_client_ids)].copy()
    else:
        messages_df = messages_df.iloc[0:0].copy() if isinstance(messages_df, pd.DataFrame) else pd.DataFrame()

    if 'created_on' in notes_df.columns:
        notes_df['created_on'] = pd.to_datetime(notes_df['created_on'], errors='coerce', utc=True)
        notes_df = notes_df.sort_values('created_on', kind='stable')

    if 'created_on' in activities_df.columns:
        activities_df['created_on'] = pd.to_datetime(activities_df['created_on'], errors='coerce', utc=True)
        activities_df = activities_df.sort_values('created_on', kind='stable')

    if 'sent_on' in messages_df.columns:
        messages_df['sent_on'] = pd.to_datetime(messages_df['sent_on'], errors='coerce', utc=True)
        messages_df = messages_df.sort_values('sent_on', kind='stable')

    invoice_data = invoice_data_df.to_csv(index=False)

    if {'invoice_number', 'created_on', 'content'}.issubset(notes_df.columns):
        notes_csv = notes_df[['invoice_number', 'created_on', 'content']].to_csv(index=False)
    else:
        notes_csv = notes_df.to_csv(index=False)

    activities_csv = activities_df.to_csv(index=False)

    message_cols = ['client_id', 'sent_on', 'subject', 'body']
    for col in message_cols:
        if col not in messages_df.columns:
            messages_df[col] = pd.NA
    messages_csv = messages_df[message_cols].to_csv(index=False)

    formatted_text = f"""
    ### INVOICE DATA:
    {invoice_data}

    ### NOTES:
    {notes_csv}

    ### MESSAGES:
    {messages_csv}

    ### ACTIVITIES:
    {activities_csv}
    """

    return formatted_text, notes_csv, invoice_data, activities_csv, messages_csv


In [36]:
def split_internal_context(internal_context, max_chars=7000, overlap_chars=350):
    text = (internal_context or "").strip()
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0
    step = max_chars - overlap_chars
    if step <= 0:
        step = max_chars

    while start < len(text):
        end = min(start + max_chars, len(text))
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += step
    return chunks


def compress_internal_context(internal_context, target_words=200, chunk_chars=7000, overlap_chars=350):
    if not internal_context or not internal_context.strip():
        return ""

    chunks = split_internal_context(internal_context, max_chars=chunk_chars, overlap_chars=overlap_chars)
    if not chunks:
        return ""

    per_chunk_words = max(60, min(140, max(1, target_words // max(1, len(chunks)))))
    chunk_summaries = []

    for idx, chunk in enumerate(chunks, start=1):
        chunk_input = f"Chunk {idx}/{len(chunks)}:\n{chunk}"
        response = internal_context_compressor(
            internal_context=chunk_input,
            target_words=per_chunk_words,
        )
        summary = response.compressed_notes.strip()
        if summary:
            chunk_summaries.append(summary)

    if not chunk_summaries:
        return ""

    merged_context = "\n".join(chunk_summaries)
    final_response = internal_context_compressor(
        internal_context=merged_context,
        target_words=target_words,
    )
    compressed = final_response.compressed_notes.strip()

    if len(compressed.split()) > target_words:
        compressed = internal_context_compressor(
            internal_context=compressed,
            target_words=target_words,
        ).compressed_notes.strip()

    if len(compressed.split()) > target_words:
        compressed = " ".join(compressed.split()[:target_words]).strip()

    return compressed

In [37]:
df_notes.query("client_id == 562116").invoice_number.tolist()

[<NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 <NA>,
 '3825917',
 '3825917',
 '3851985']

In [25]:
i = 26

invoice_numbers_by_matter = df_invoices.groupby(by='matter_id').agg(list).invoice_number.tolist()
all_invoice_numbers_list = [lst for lst in invoice_numbers_by_matter if len(lst) >= 2]

invoice_numbers = all_invoice_numbers_list[i]

formatted_data, notes, invoice_data, activities, messages = data_formatter_csv_list_v2(
    df_invoices,
    df_notes,
    df_activities,
    df_messages,
    invoice_numbers,
)

internal_context = f"""### NOTES:
{notes}

### ACTIVITIES:
{activities}"""
email_conversation = messages if isinstance(messages, str) else ""

print(invoice_numbers)
print(formatted_data)



['3777397', '3804854', '3826948', '3849897', '3872548']

    ### INVOICE DATA:
    invoice_number,invoice_amount,ar_amount,submitted_on,due_on,days_overdue,status
3777397,10962.0,10962.0,2025-07-17 21:02:15.076653+00:00,2025-08-16 00:00:00+00:00,126,Delivered
3804854,1249.5,1249.5,2025-09-05 21:21:30.339663+00:00,2025-10-05 00:00:00+00:00,76,Delivered
3826948,1191.0,691.0,2025-10-10 21:20:50.025734+00:00,2025-11-09 00:00:00+00:00,41,Delivered
3849897,845.5,595.5,2025-11-13 20:46:48.289659+00:00,2025-12-13 00:00:00+00:00,7,Delivered
3872548,11816.5,11816.5,2025-12-15 22:42:21.587708+00:00,2026-01-14 00:00:00+00:00,0,Delivered


    ### NOTES:
    invoice_number,created_on,content


    ### MESSAGES:
    client_id,sent_on,subject,body
567518,2025-08-28 12:34:41.720362+00:00,RE: **REPLY REQUESTED** <Organization>/<Organization> (0567518) <Person>,"Good morning <Person> Could I please ask you to make the first $250.00 payment by tomorrow? To pay online in minutes, you may visit our payment

In [26]:
email_conversation


'client_id,sent_on,subject,body\n567518,2025-08-28 12:34:41.720362+00:00,RE: **REPLY REQUESTED** <Organization>/<Organization> (0567518) <Person>,"Good morning <Person> Could I please ask you to make the first $250.00 payment by tomorrow? To pay online in minutes, you may visit our payment portal at <URL> Thank <Person>\u200b\u200b\u200b\u200bRevenue Cycle Operations AdvisorDirect: 312-526-1575Mobile: 616‑-745‑<Email> From: <Person> <<Email>> Sent: Thursday, August 14, 2025 7:30 AMTo: <Email>: RE: **REPLY REQUESTED** Husch Blackwell<Organization> (0567518) <Person> Good morning I\'m more than happy to discuss by phone, and I am mostly available <DateTime> and <DateTime>. Please feel free to contact me at either number in my signature line. Thank <Person>\u200b\u200b\u200b\u200bRevenue Cycle Operations <Organization> Link Virtual <Address>3912Direct: 312-526-1575Mobile: 616‑-745‑-7798Fax: <Email> <Organization> is a different kind of law firm—structured around our clients\' industries a

In [ ]:
with dspy.context(lm=lm_41_mini.copy(cache=False, temperature=0.2)):
    response = summarizer(
        financial_data=invoice_data,
        notes=notes,
        email_conversation=email_conversation,
    )
    print(response)



Prediction(
    summary='The client currently has five outstanding invoices totaling approximately $25,314, with the oldest invoice #3777397 overdue by 126 days as of Dec 16, 2025, and the most recent invoice #3872548 just issued on Dec 15, 2025. Recent email exchanges in late August 2025 show the client acknowledged the overdue balance and expressed willingness to set up a payment plan, making a partial payment of $250 after a request for immediate payment by Aug 22, 2025. Earlier communications reveal ongoing service provision despite no payments since February 2025, with the client open to phone discussions to arrange manageable payments. Next steps should focus on confirming and formalizing a payment plan to reduce the significant overdue amounts and monitoring payments on the newer invoices to prevent further aging.'
)


In [31]:
print(invoice_data)

invoice_number,invoice_amount,ar_amount,submitted_on,due_on,days_overdue,status
3777397,10962.0,10962.0,2025-07-17 21:02:15.076653+00:00,2025-08-16 00:00:00+00:00,126,Delivered
3804854,1249.5,1249.5,2025-09-05 21:21:30.339663+00:00,2025-10-05 00:00:00+00:00,76,Delivered
3826948,1191.0,691.0,2025-10-10 21:20:50.025734+00:00,2025-11-09 00:00:00+00:00,41,Delivered
3849897,845.5,595.5,2025-11-13 20:46:48.289659+00:00,2025-12-13 00:00:00+00:00,7,Delivered
3872548,11816.5,11816.5,2025-12-15 22:42:21.587708+00:00,2026-01-14 00:00:00+00:00,0,Delivered



In [29]:
with dspy.context(lm=lm_41_mini.copy(cache=False, temperature=0.2)):
    response = summarizer_cot(
        financial_data=invoice_data,
        notes=notes,
        email_conversation=email_conversation,
    )
    print(response)



Prediction(
    reasoning='The client currently has five invoices with a total outstanding balance of $24,314.50, with the oldest invoice (3777397) overdue by 126 days and the most recent invoice (3872548) just issued with no overdue days. The email conversation from August 2025 reveals the client had not made payments since February and was requested to make an immediate payment by August 22, 2025, to bring the balance current. The client responded positively, expressing willingness to set up a payment plan with small payments and confirmed a partial payment of $250 was made by August 28, 2025. There are no notes to provide additional context. The pattern suggests partial payments are being made but the balance remains significantly overdue, indicating the need for continued monitoring and follow-up to ensure the payment plan progresses and the balance reduces.',
    summary='The client has an outstanding balance of **$24,314.50 across five invoices**, with the oldest invoice overdue 

In [28]:
compressed_notes = compress_internal_context(internal_context, target_words=200)
notes_for_summary = compressed_notes if compressed_notes else notes
with dspy.context(lm=lm_41_mini.copy(cache=False, temperature=0.4)):
    response = summarizer_cot(
        financial_data=invoice_data,
        notes=notes_for_summary,
        email_conversation=email_conversation,
    )



NameError: name 'internal_context_compressor' is not defined

In [45]:
print(compressed_notes)

Client has $16,373 overdue AR across 3 active invoices with an average DSO of 62. September and October invoices remain unpaid despite payments on November and January. Repeated emails from late 2024 to early 2026 addressed payment status and invoice corrections due to address issues. Contact updates occurred with new owners assigned. Client cited multi-level approval delays but promised payment; follow-ups planned if December invoice remains unpaid by 3/15/2026. Separately, a $16,816.29 balance from August to December 2023 is unpaid with no payments received. Multiple emails sent to internal contacts from October to December 2023; client described as "really scattered," but internal contacts committed to pressing client after receiving documents. Client agreed to check payment status; follow-up outreach ongoing with no confirmed payments yet.


In [46]:
response

Prediction(
    reasoning='The client currently has $16,373 in overdue accounts receivable across three active invoices, contributing to an average DSO of 62 days. Despite payments in November and January, the September and October invoices remain unpaid, indicating partial but inconsistent payment behavior. Multiple communications from late 2024 through early 2026 have focused on payment status, invoice corrections, and client contact updates, with the client citing multi-level approval delays but promising payment. There is also a separate older balance of $16,816.29 from August to December 2023 that remains unpaid, with ongoing follow-ups but no confirmed payments. The client is described as "really scattered," and internal contacts have committed to pressing the client further. Follow-up is planned if the December invoice remains unpaid by March 15, 2026, indicating a need for continued monitoring and outreach.',
    summary='The client has **$16,373 overdue across three active inv

In [ ]:
from tqdm import tqdm
import time

invoice_numbers_by_matter = df_invoices.groupby(by='matter_id').agg(list).invoice_number.tolist()
all_invoice_numbers_list = [lst for lst in invoice_numbers_by_matter if len(lst) >= 2]

for i in tqdm(range(len(all_invoice_numbers_list))):
    invoice_numbers = all_invoice_numbers_list[i]
    formatted_data, notes, invoice_data, activities, messages = data_formatter_csv_list_v2(
        df_invoices,
        df_notes,
        df_activities,
        df_messages,
        invoice_numbers,
    )

    internal_context = f"""### NOTES:
{notes}

### ACTIVITIES:
{activities}"""
    compressed_notes = compress_internal_context(internal_context, target_words=200)
    notes_for_summary = compressed_notes if compressed_notes else notes
    email_conversation = messages if isinstance(messages, str) else ""

    with dspy.context(lm=lm_41_mini.copy(cache=False, temperature=0.4)):
        response = summarizer_cot(
            financial_data=invoice_data,
            notes=notes_for_summary,
            email_conversation=email_conversation,
        )
    time.sleep(5)



In [ ]:
with dspy.context(lm=lm_4o_mini.copy(cache=False, temperature=0.4)):
    response = summarizer(
        financial_data=invoice_data,
        notes=compressed_notes,
        email_conversation=email_conversation,
    )
    print(response)



In [ ]:
with dspy.context(lm=lm_41_mini.copy(cache=False, temperature=0.4)):
    response = summarizer(
        financial_data=invoice_data,
        notes=compressed_notes,
        email_conversation=email_conversation,
    )
    print(response)



In [ ]:
with dspy.context(lm=lm_5_mini):
    response = summarizer_cot(
        financial_data=invoice_data,
        notes=compressed_notes,
        email_conversation=email_conversation,
    )
    print(response)

